# Flight Trend & Optimization Analyzer

## 1. Project Overview

This project fetches **real-time global flight data** from the [OpenSky Network API](https://opensky-network.org) and performs data analysis using Object-Oriented Programming (OOP) in Python.

**Core objectives:**
1. Fetch live flight data via REST API
2. Analyze country distribution and airline activity
3. Study velocity and altitude patterns
4. Filter and export data for further use


---
## 2. Environment Setup & Module Import


In [1]:
import pandas as pd
from src.flight_logic import FlightDataAnalyzer, LiveFlightAPI

print("[INFO] Environment initialized. Modules loaded: FlightDataAnalyzer, LiveFlightAPI.")


[INFO] Environment initialized. Modules loaded: FlightDataAnalyzer, LiveFlightAPI.


---
## 3. Load Live Data from OpenSky Network API

We use the `LiveFlightAPI` child class (inherits from `FlightDataAnalyzer`) to fetch real-time flight data.
The endpoint `https://opensky-network.org/api/states/all` delivers a dynamic **JSON response**:
- The root object contains `time` (Unix timestamp) and `states` (a 2D array of airborne aircraft state vectors).
- Because `states` contains an array of raw values rather than named keys, our ingestion logic parses the official index positions:
  - `flight[0]`: **icao24** (unique 24-bit transponder hex ID)
  - `flight[1]`: **callsign** (8-character flight identifier)
  - `flight[2]`: **country** (aircraft country of registration)
  - `flight[5]`: **longitude** & `flight[6]`: **latitude** (WGS-84 coordinates)
  - `flight[7]`: **altitude** (barometric altitude in meters)
  - `flight[9]`: **velocity** (ground speed in m/s)
  - `flight[10]`: **heading** (direction of motion in degrees)

The parsed records are converted into a `pandas.DataFrame` and cleaned of missing transponder records.


In [2]:
# Initialize API loader and fetch live flight data
live_api = LiveFlightAPI()

if live_api.fetch_live_flights(limit=500):
    print(f"[SUCCESS] Loaded {len(live_api.df)} live flights from OpenSky Network")
    print(f"Columns: {list(live_api.df.columns)}")
    display(live_api.df.head(10))
else:
    print("[ERROR] Failed to fetch data. Check network connection.")


[SUCCESS] Loaded 500 live flights from OpenSky Network
Columns: ['icao24', 'callsign', 'country', 'longitude', 'latitude', 'altitude', 'velocity', 'heading']


,icao24,callsign,country,longitude,latitude,altitude,velocity,heading
0,39de4f,TVF33VV,France,NaN,NaN,NaN,169.46,343.60
1,a09284,XAA9854,United States,-74.8760,39.3853,7315.20,218.68,220.13
2,e8027c,LPE2401,Chile,-77.1841,-11.9045,662.94,90.42,154.02
3,80162d,AXB842,India,51.5623,26.6250,7658.10,199.98,109.69
4,801638,AXB1823,India,76.9509,28.2388,4792.98,187.38,246.88
5,ac494e,CMD3,United States,-121.1567,39.0438,746.76,71.49,327.34
6,3d10af,DEEOL,Germany,15.4257,46.2297,3360.42,68.78,313.48
7,39de4b,TVF44UX,France,20.8060,45.8147,11582.40,211.93,310.77
8,39de4c,TVF21CH,France,8.3278,47.3477,11277.60,248.06,126.82
9,39de57,TVF15CC,France,2.0800,41.2949,NaN,6.94,64.69


---
## 4. Feature 1: Country Distribution Analysis

Which countries have the most airborne flights right now?


In [3]:
# Top 10 countries by live flight count (tabular analysis)
country_stats = live_api.df["country"].value_counts().head(10)

print("=== Top 10 Countries by Live Flight Count ===")
for country, count in country_stats.items():
    print(f"  {country}: {count} flights")


=== Top 10 Countries by Live Flight Count ===
  United States: 80 flights
  India: 61 flights
  Sweden: 48 flights
  Hungary: 32 flights
  Austria: 28 flights
  Turkey: 24 flights
  Germany: 23 flights
  Greece: 20 flights
  Spain: 19 flights
  France: 17 flights


---
## 5. Feature 2: Velocity & Altitude Analysis

Analyze flight speed and altitude patterns across countries.


In [4]:
# Compute velocity and altitude stats by country
valid_flights = live_api.df.dropna(subset=["velocity", "altitude"])

vel_stats = valid_flights.groupby("country").agg(
    flight_count=("icao24", "count"),
    avg_velocity=("velocity", "mean"),
    avg_altitude=("altitude", "mean"),
    max_velocity=("velocity", "max")
).reset_index()

vel_stats["avg_velocity"] = vel_stats["avg_velocity"].round(2)
vel_stats["avg_altitude"] = vel_stats["avg_altitude"].round(2)
vel_stats["max_velocity"] = vel_stats["max_velocity"].round(2)

top_vel = vel_stats.sort_values("flight_count", ascending=False).head(10)
print("=== Top 10 Countries: Velocity & Altitude Stats ===")
display(top_vel)


=== Top 10 Countries: Velocity & Altitude Stats ===


,country,flight_count,avg_velocity,avg_altitude,max_velocity
35,United States,64,189.94,7409.14,301.37
16,India,60,196.56,7403.46,266.45
30,Sweden,39,210.78,9464.43,251.64
14,Hungary,30,181.92,7931.91,248.71
2,Austria,27,191.68,8656.88,250.06
33,Turkey,23,247.62,9951.39,1093.17
12,Germany,20,144.55,5617.84,258.89
29,Spain,16,214.48,9713.12,251.53
13,Greece,16,185.97,7067.07,259.13
11,France,15,171.95,7592.06,251.71


---
## 6. For-loop: Detailed Flight Summary by Country

Use explicit `for` loops to iterate and print detailed stats for each top country.


In [5]:
# For-loop: print detailed summary for top countries
top_n = 5
country_counts = live_api.df["country"].value_counts().head(top_n)

print(f"\n{'='*60}")
print(f"  Top {top_n} Countries - Detailed Flight Summary")
print(f"{'='*60}")

for country, count in country_counts.items():
    subset = live_api.df[live_api.df["country"] == country]
    avg_vel = subset["velocity"].mean() if not subset["velocity"].isna().all() else 0
    avg_alt = subset["altitude"].mean() if not subset["altitude"].isna().all() else 0
    print(f"\nCountry: {country}")
    print(f"  Total flights: {count}")
    print(f"  Avg velocity: {avg_vel:.2f} m/s")
    print(f"  Avg altitude: {avg_alt:.2f} m")

print(f"\n{'='*60}")



  Top 5 Countries - Detailed Flight Summary

Country: United States
  Total flights: 80
  Avg velocity: 152.93 m/s
  Avg altitude: 7409.14 m

Country: India
  Total flights: 61
  Avg velocity: 193.41 m/s
  Avg altitude: 7403.47 m

Country: Sweden
  Total flights: 48
  Avg velocity: 174.76 m/s
  Avg altitude: 9464.43 m

Country: Hungary
  Total flights: 32
  Avg velocity: 170.60 m/s
  Avg altitude: 7931.91 m

Country: Austria
  Total flights: 28
  Avg velocity: 184.84 m/s
  Avg altitude: 8656.88 m



In [ ]:
live_api.show_for_loop_demo()
